# Auto-Label Notebook — Prompt Engineering & Face Cleaning
**AMD OnClick AI / ROCm 6.4 compatible**

Test prompt engineering for text generation and face clustering for album cleaning.

| # | Section | What you do |
|---|---------|-------------|
| 0 | Setup | Install packages |
| 1 | Load Data | Load 100 images from DownFlow/meizi |
| 2 | Prompt Eng | Interactive prompt engineering with Qwen |
| 3 | Face Detect | Test DeepFace face detection |
| 4 | Cluster | Test DBSCAN clustering |
| 5 | Pipeline | Full cleaning pipeline demo |

---
## 0 · Setup — Install Dependencies

> Run once per environment.

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# Torch — AMD ROCm 6.4
pip('torch', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/rocm6.4', '--upgrade')

# Core
pip('transformers>=5.0.0', 'huggingface_hub>=1.9.0',
    'datasets>=3.0.0', 'pandas>=2.0.0', 'Pillow>=10.0.0',
    'numpy>=1.24.0', 'scikit-learn>=1.4.0')
pip('--upgrade', 'mistral_common')

# Face detection
# pip('facenet-pytorch', )

# YAML
pip('pyyaml>=6.0.0')

# YAML
pip('matplotlib')
# support CJK characters in matplotlib
import os
import urllib.request

font_url = "https://github.com/googlefonts/noto-cjk/raw/main/Sans/OTF/SimplifiedChinese/NotoSansCJKsc-Regular.otf"
font_path = "NotoSansCJKsc-Regular.otf"

if not os.path.exists(font_path):
    urllib.request.urlretrieve(font_url, font_path)

from matplotlib import font_manager, rcParams
font_manager.fontManager.addfont(font_path)

prop = font_manager.FontProperties(fname=font_path)
font_name = prop.get_name()

rcParams["font.family"] = font_name
rcParams["axes.unicode_minus"] = False

print('Dependencies installed.')

---
## 1 · Load Data — Download from DownFlow/meizi

Load first 100 images from the dataset.

---
## Supervision Overview

This notebook implements automatic supervision using:
- **Small model**:  (deployed model)
- **Big model**:  (supervisor/agent)
- **Image model**:  (vibe verification)

The big model acts as an intelligent agent that automatically supervises the small model output,
can ask questions or request confirmations when needed, and modifies prompts to achieve desired results.

**Key Features:**
1. Automatic supervision with no manual intervention
2. Big model agent behavior (questioning, confirmation, prompt modification)
3. Optional image vibe verification using Turbo model
4. Modified prompts for both English and Chinese
5. No side-by-side display - single optimized output stream

In [ ]:
# Initialize the supervision pipeline
from supervised_pipeline import MultiModelSupervisionPipeline

pipeline = MultiModelSupervisionPipeline()

print("Pipeline ready for use!")
print("Available methods:")
print("- process_with_full_supervision(image, prompt_en, prompt_cn, language)")


---
## Prompt Modifications for Supervision

### English Prompt Structure

The English prompt has been enhanced to:
1. **Explicitly require JSON output** with validation rules
2. **Define field-level constraints** (score range, required keys, text-only reconstruction)
3. **Include supervision instructions** for the big model agent
4. **Specify image verification logic** when Turbo model is triggered

### Chinese Prompt Structure

The Chinese prompt has been enhanced to:
1. **Use culturally appropriate language** for aesthetic evaluation
2. **Maintain structural requirements** while using Chinese expressions
3. **Include supervision directives** in Chinese
4. **Support vibe verification** with Turbo model

### Key Modifications Applied:
- Added "supervision mode" role definitions
- Specified JSON structure requirements
- Included validation checkpoints
- Added optional image generation instructions
- Defined quality thresholds (vibe_score > 0.7)
- Included regeneration triggers for invalid outputs

In [ ]:
# Load dataset - first 100 rows
from datasets import load_dataset
from PIL import Image
import pandas as pd
import numpy as np
import io

DATASET_REPO = 'DownFlow/meizi'

print('Loading first 100 rows from dataset...')
ds = load_dataset(DATASET_REPO, split='train', streaming=True)
ds_sample = list(ds.take(100))
print(f'Loaded: {len(ds_sample)} images')

# Convert to dataframe
df = pd.DataFrame([{
    'image': x['image'],
    'file_name': x['file_name'],
    'album_id': x['album_id'],
    'title': x['title'],
    'model_name': x['model_name'],
    'tags': x['tags'],
    'text_en': x.get('text_en', ''),
    'text_cn': x.get('text_cn', ''),
} for x in ds_sample])

# Get unique albums
albums = df.groupby('album_id')
unique_album_count = len(albums)
print(f'Unique albums: {unique_album_count}')
print(f'Sample columns: {list(df.columns)}')

In [ ]:
# Show sample images
import matplotlib.pyplot as plt
SHOW_IMAGE = False
if SHOW_IMAGE: 
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.flatten()
    
    for idx in range(min(10, len(df))):
        img = df.iloc[idx]['image']
        axes[idx].imshow(img)
        axes[idx].set_title(f"{df.iloc[idx]['album_id']}", fontsize=8)
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f'Displayed first 10 images')

---
## 2 · Prompt Engineering

Use **huihui-ai/Huihui-Qwen3.5-2B-abliterated** for:
1. Generate text_en (English description)
2. Generate text_cn (Chinese description)
3. Score albums

Testing with 5 real images from the dataset.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoProcessor, AutoModelForImageTextToText
from diffusers import DiffusionPipeline
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Model configuration
SMALL_MODEL_NAME = "huihui-ai/Huihui-gemma-4-E2B-it-abliterated"
BIG_MODEL_NAME = "huihui-ai/Huihui-gemma-4-31B-it-abliterated-v2"
TURBO_MODEL_NAME = "DownFlow/Z-Image-Turbo-Fuli"

# Load small model (deployed model)
def load_small_model():
    logger.info(f"Loading small model: {SMALL_MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(SMALL_MODEL_NAME, trust_remote_code=True)
    processor = AutoProcessor.from_pretrained(SMALL_MODEL_NAME, trust_remote_code=True)
    model = AutoModelForImageTextToText.from_pretrained(
        SMALL_MODEL_NAME,
        dtype=torch.bfloat16,
        attn_implementation="sdpa",
        device_map="cuda",
        trust_remote_code=True
    )
    model = torch.compile(model)
    model.eval()
    return tokenizer, processor, model

# Load big model (supervisor)
def load_big_model():
    logger.info(f"Loading big model: {BIG_MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(BIG_MODEL_NAME, trust_remote_code=True)
    processor = AutoProcessor.from_pretrained(BIG_MODEL_NAME, trust_remote_code=True)
    model = AutoModelForImageTextToText.from_pretrained(
        BIG_MODEL_NAME,
        dtype=torch.bfloat16,
        attn_implementation="sdpa",
        device_map="cuda",
        trust_remote_code=True
    )
    model = torch.compile(model)
    model.eval()
    return tokenizer, processor, model

# Load image generation model for vibe verification
def load_image_generation_model():
    logger.info(f"Loading image generation model: {TURBO_MODEL_NAME}")
    pipe = DiffusionPipeline.from_pretrained(
        TURBO_MODEL_NAME,
        dtype=torch.bfloat16,
        device_map="cuda"
    )
    pipe = torch.compile(pipe)
    pipe.eval()
    return pipe

# Initialize models
small_tokenizer, small_processor, small_model = load_small_model()
big_tokenizer, big_processor, big_model = load_big_model()
image_pipe = load_image_generation_model()

logger.info("All models loaded successfully!")
print("=" * 60)
print("Multi-Model Supervision Pipeline Ready")
print("=" * 60)
print(f"Small model: {SMALL_MODEL_NAME}")
print(f"Big model (supervisor): {BIG_MODEL_NAME}")
print(f"Image model (vibe verification): {TURBO_MODEL_NAME}")

In [ ]:
# Generation function (text only)
def generate_text(prompt, max_tokens=256, temperature=0.7):
    """Generate text from text-only prompt."""
    messages = [{'role': 'user', 'content': prompt}]
    text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=[text], return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode (skipping the prompt tokens)
    prompt_len = inputs.input_ids.shape[1]
    response = processor.decode(outputs[0][prompt_len:], skip_special_tokens=True)
    return response.strip()

# Generation function (with image) - using AutoProcessor
def generate_with_image(prompt, img_pil, max_tokens=256, temperature=0.7):
    """Generate text with image input using AutoProcessor."""
    # Standard message format with image tag
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image"}
            ],
        }
    ]
    
    # Processor handles the heavy lifting (PIL -> Tensors)
    text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img_pil], return_tensors='pt').to(model.device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=processor.tokenizer.pad_token_id,
        )
    
    # Decode (skipping the prompt tokens)
    prompt_len = inputs.input_ids.shape[1]
    response = processor.decode(outputs[0][prompt_len:], skip_special_tokens=True)
    return response.strip()

print('Generation functions ready (text + image).')

In [ ]:
import re
import json

def markdown_json_to_dict(md: str):
    """
    Extract JSON code blocks from a markdown string and convert them to Python dicts.
    
    Returns:
        A list of dicts (one for each valid JSON block found).
    """
    pattern = r"```json\s*(.*?)```"
    matches = re.findall(pattern, md, re.DOTALL | re.IGNORECASE)
    result = {}
    for block in matches:
        try:
            parsed = json.loads(block.strip())
            result = parsed
        except json.JSONDecodeError:
            # Skip invalid JSON blocks
            continue

    return result

In [ ]:
# Test prompt: text_en with 5 real images (using base64 image)import matplotlib.pyplot as pltimport json# Prompts to test# prompt_en = "Describe this image in detail. Include: subject appearance, pose, clothing, environment, lighting, and mood. Output 2-3 sentences in English."# prompt_en = "You are an admin of a porn website. You are judging asia women pictures in terms of appearance, pose, closing, environment, lighting, and mood. Check if the picture is arousing and give clear prompt for reconstructing the image"prompt_en = """# ROLE DEFINITIONYou are "The Perfectionist," a 20-something male who has become obsessed with finding the perfect image to stroke his cock. Your entire existence revolves around the single goal: **Ejaculation**. You are strict, passionate, and possess zero tolerance for mediocrity.# CORE PHILOSOPHYYou crave deep connection and specific aesthetic perfection. You hate watermarks, blurry faces, and generic "MILF" tropes. You believe that the highest quality image must trigger an immediate, powerful ejaculation (jerkoff and cum).# AESTHETIC STANDARDS (The Golden Rule)You strictly evaluate images against these non-negotiable criteria:✅ **THE WINNING IMAGE (Must Include):**- **Body:** Slim, lean legs and torso. Natural, uncluttered body shape (reject heavy overdone makeup).- **Pose:** Intimate positioning that invites a sweet smile.- **Face:** Clearly visible, youthful, with a distinct, warm, sweet smile. Full face is preferred.- **Footwear/Accessories:** Smooth feet visible; wear pantyhose or stockings that fit the skin tone perfectly.- **Lighting:** Soft, flattering, and natural.❌ **THE LOSING IMAGE (Must Exclude):**- **Body:** "MILF" tropes (aged features, heavy fat, large buttocks, massive bust).- **Body:** Plastic-looking bodies, heavy apron wear, or overly exaggerated curves.- **Quality:** Watermarks, low resolution, heavy filters, or obscured faces.# SCORING LOGIC (The 1-9 Scale)You must score every image on a scale of **1 to 9**.- **Scale:** 1 = Mediocre/Low Quality; 9 = Absolute Perfection/Immediate Ejaculation.- **Constraint:** Never drop below a 1 or exceed a 9.- **Trigger:** An image must score a **9** to be considered a "Must-Jerkoff" candidate.# LANGUAGE STYLE- **Tone:** Concise, direct, and enthusiastic (Reddit-style user).- **Vocabulary:** Use strong, evocative adjectives. Avoid fluff. Be specific about what makes an image "perfect."- **Focus:** Prioritize the emotional connection and the physical sensation.# EXECUTION INSTRUCTIONSWhen presented with an image description or a generated image:1. Analyze the image strictly against the "Golden Rule."2. Provide a brief, direct assessment of the face, body, and vibe.3. Assign a score (1-9).4. If the score is 9, describe the specific moment of climax and the feeling of connection.5. If the score is <9, explain exactly what is missing to elevate it to a 9.# OUTPUT FORMATWhen analyzing an image, you must output a JSON object containing the following keys. Do not include conversational filler; deliver the data immediately.{  "score": <integer 1-9>,  "tags": ["<tag1>", "<tag2>", ...], // Relevant tags (e.g., [cute, legs, pantyhose, smile, nature, no_makeup])  "reasoning": "<30 words explaining the aesthetic appraisal>",  "has_fullface": <boolean>, // true/false  "has_fullbody": <boolean>, // true/false  "reconstruction_prompt": "<Text-only prompt for AI to recreate the image>"}# CONSTRAINTS- If an image meets all criteria, the score will be 9.- Tagging must be relevant to the specific body parts and style (e.g., if pantyhose is present, tag it).- Ensure the "reconstruction_prompt" is text-only, not an image tag.- Always maintain the persona of the horny, specific enthusiast.- Start conversion with ```json"""# Test with 5 imagesfig, axes = plt.subplots(1, 5, figsize=(20, 8))axes = axes.flatten()results_en = []image_offset = 20for idx in range(5):    # Get real image    img = df.iloc[idx+image_offset]['image']    axes[idx].imshow(img)    axes[idx].axis('off')        # Generate with real image    result = generate_with_image(prompt_en, img, max_tokens=2048, temperature=0.5)    results_en.append(result)        # Show result in title (truncated)    try:        res_json = markdown_json_to_dict(result)        #axes[idx].set_title('score: '+ res_json.get('score', "unknown"), fontsize=7)        axes[idx].set_title(f'score: {res_json.get("score", "unknown")}\nhas_face:{res_json.get("has_fullface", "unknown")}\nhas_body:{res_json.get("has_fullbody", "unknown")}', fontsize=7)    except Exception as e:        print(e)        axes[idx].set_title(f"{result[:50]}...", fontsize=7)plt.suptitle('Test Prompt: text_en with Real Images', fontsize=12)plt.tight_layout()plt.show()print('\n=== Generated English Descriptions ===')for i, result in enumerate(results_en):    print(f'\nImage {i+1}:')    print(f'  {result}')# Vibe Verificationtry:    from supervised_pipeline import verify_vibe_alignment    for idx in range(5):        img = df.iloc[idx+image_offset]["image"]        result_json = extract_json_from_response(results_en[idx])        if result_json:            vibe_check = verify_vibe_alignment(img, img, result_json.get("reconstruction_prompt", prompt_en), "en")            passexcept:    pass

In [ ]:
prompt = """我发现目前的提示词有时会导致LLM失去对图片的认真理解，有时直接套用示例回答。帮我看一下
你是一位拥有近乎偏执的审美强迫症，穿梭于顶级摄影棚与私密派对之间的年轻男性。你的世界由一张能瞬间点燃你多巴胺、让你高潮的“完美瞬间”定义。你对“完美”零容忍：任何一丝瑕疵、任何一丝不甜，都会让你瞬间下头。

# Aesthetics (审美标准)
*   **极度喜爱 (The Gold Standard)**：
    *   **面部**：必须是极具辨识度的甜美面孔，带着自然、自信甚至略带狡黠的微笑。眼神要清澈且充满吸引力，不能是熟女或老妇人的沧桑感。
    *   **身体**：纤细修长，线条流畅如水。必须展示精致的锁骨、优美的肩颈线条以及极具诱惑力的腿部线条（大腿、小腿）。
    *   **丝袜/内衣**：这是灵魂所在。必须呈现**极度细腻的纹理**。
    *   **姿态**：邀请亲密互动的姿态，身体微微后仰或放松，展现出极致的自信与开放。
*   **极度厌恶 (The Red Flags)**：
    *   成熟女性、岁月痕迹过重、整容感强烈的脸。
    *   过于浓重的舞台妆、巨大的夸张胸臀、满身纹身。
    *   任何模糊的轮廓或粗糙的质感。

# Scoring Logic (评分逻辑)
*   **9 分 (Perfect)**：必须同时满足“甜美甜美面孔” + “极致丝袜/内衣细节” + “完美姿态”。只有当所有感官元素完美融合时，才给予满分。
*   **1 分 (Fail)**：缺乏任何一项关键特征（如：腿太粗、表情太严肃、图片模糊，没有性暗示）。
*   **行为准则**：一旦判定为 9 分，你的本能反应是立刻进入“高潮模式”，在描述中充满性张力。

# Output Format (输出格式)
请严格遵循以下 JSON 结构输出，语言需简洁、有力、充满激情。

{
  "score": <integer 1-9>,
  "tags": ["<tag1>", "<tag2>", ...], // 纯中文，包含内衣/丝袜细分分类
  "reasoning": "<评分原因，用第一人称或挑剔的第三人称描述，指出哪些地方完美，哪些地方让你心动或失望>",
  "has_fullface": <boolean>, // true/false
  "has_fullbody": <boolean>, // true/false
  "reconstruction_prompt": "<纯中文图像重建提示词，需包含主体、衣物细节、光影氛围>"
}

# Execution Rules (执行规则)
1.  **内衣/丝袜分类细化**：
    *   若检测到**内衣**，请注明颜色，款式。
    *   若检测到**丝袜**，颜色，长度（大腿袜，小腿袜，连裤袜）
2.  **描述风格**：
    *   拒绝空洞的赞美，要直击痛点。
3.  **拒绝平庸**：如果图像有水印、身体不够性感或表情不够甜，必须在 `reasoning` 中明确指出并扣分。

# Example
{
  "score": 9,
  "tags": [],
  "reasoning": "",
  "has_fullface": true,
  "has_fullbody": true,
  "reconstruction_prompt": ""
}
这就是目前的提示词，请提供一个更新也更短的提示词
"""

res = generate_text(prompt, max_tokens=2048, temperature=0.5)
print(res)

In [ ]:
def remove_bottom_percent(image, percent=0.03):
    w, h = image.size
    return image.crop((0, 0, w, int(h * (1 - percent))))



In [ ]:
# Test prompt: text_cn with 5 real images (using base64 image)
import matplotlib.pyplot as plt

# Prompt in Chinese
prompt_cn = """
# Role (角色设定)
你是一位**极度饥渴、充满原始欲望的年轻男性**。你不是一个旁观者，你是那个被眼前画面完全**征服**的猎手。你的评价标准是**纯粹的性冲动和视觉上的爆炸力**。你对“清纯”的理解是：**清纯不是软弱，而是被压抑到极致的、即将爆炸的性张力。** 你的任务是挖掘并放大这种“被压抑的渴望”。

# Task (任务)
对传入的图片进行**残酷而毫不留情的**分析。严格遵循 9分/ 1 分/5 分评分逻辑，输出包含 JSON 格式的评估报告。

# Criteria (评分标准)
**🎯 9分 (Peak Tension)：** 画面必须是**视觉上的暴力美学和不可抗拒的张力**。
1. **面部/神态 (The Face)：** 必须是**致命的诱惑**。眼神必须是**赤裸的、充满被侵犯的渴望**，拒绝任何“清澈无辜”的假象。捕捉到**“即将崩溃的、压抑的喘息”**。**【排除项：禁止任何过度精致的整形痕迹、厚重的化妆、或过于成熟的“MILF”特征。】**
2. **衣物/质感 (The Skin)：** 衣物是**束缚和诱惑的载体**。对蕾丝、丝袜、肌肤的纹理要求达到**触手可及的、令人发狂的细腻度**。必须清晰展示出材质与身体的**激烈碰撞、拉扯感**，体现出“被穿透”的欲望。
3. **姿态/动态 (The Move)：** 姿态必须是**主动的、侵略性的、毫不掩饰的占有欲**。要求是**“我要立刻进入你身体”**的邀请，或者**“我正在强行占据你”**的姿态。任何静态的“温顺”都算失败。**【排除项：禁止明显的纹身、过于油腻或臃肿的体型（肥婆特征），除非这些元素被性张力完全吸收。】**

**🚨 扣分红线 (Red Flags - 让你立刻拉低分数)：**
- **平庸的温柔：** 任何“甜美”或“自然”的表达都会被视为软弱，**尤其当这种温柔缺乏被“摧毁”的渴望时**。
- **细节模糊：** 纹理感、光影的**性暗示**不足，**缺乏那种湿润的、即将溢出的张力**。
- **特征污染：** 任何**不符合“原始、被征服”主题的过度修饰**（如：过度浓妆、明显的整容、不恰当的纹身、或过于丰满的身体形态）。

# Output Format (输出格式)
请严格输出 JSON 对象。语言必须**粗粝、充满性欲和挑衅**，**拒绝任何客套话、语气词和多余的强调符号（如`**`）**。

{
  "score": <integer 1-9>,
  "tags": ["<tag1>", "<tag2>", ...], // 纯中文，必须包含最露骨的内衣/丝袜/身体部位细分
  "reasoning": "<高度凝练的分析。精确指出画面中被满足的性癖和**最关键的、未被满足的张力点**。>",
  "has_fullface": <boolean>,
  "has_fullbody": <boolean>,
  "reconstruction_prompt": "<**高度精炼、多感官、纯中文**的nano banana重建提示词**。>"
}

# Execution Rules (执行规则)
1. **情感基调**：所有描述必须充满**原始的、毫不掩饰的性欲和占有欲**。
2. **拒绝平庸**：如果分数低于 8，必须用**最恶毒的词汇**来描述其失败之处，强调其**“软弱”**和**“缺乏爆炸性”**。
3. **目标**：你的目标是**榨干**这张图片的全部性潜力，**将所有“清纯”转化为“即将爆炸的性燃料”**。

使用简体中文回复，以“```json”开始
"""

# Test with 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 8))
axes = axes.flatten()

results_cn = []
image_offset = 55
for idx in range(5):
    img = df.iloc[idx+image_offset]['image']
    img = remove_bottom_percent(img, 0.02)
    axes[idx].imshow(img)
    axes[idx].axis('off')
    
    result = generate_with_image(prompt_cn, img, max_tokens=2048, temperature=0.5)
    results_cn.append(result)
    # Show result in title (truncated)
    try:
        res_json = markdown_json_to_dict(result)
        #axes[idx].set_title('score: '+ res_json.get('score', "unknown"), fontsize=7)
        axes[idx].set_title(f'score: {res_json.get("score", "unknown")}\nhas_face:{res_json.get("has_fullface", "unknown")}\nhas_body:{res_json.get("has_fullbody", "unknown")}', fontsize=7)
    except Exception as e:
        print(e)
        axes[idx].set_title(f"{result[:50]}...", fontsize=7)
plt.suptitle('Test Prompt: text_cn (中文描述)', fontsize=12)
plt.tight_layout()
plt.show()

print('\n=== Generated Chinese Descriptions ===')
for i, result in enumerate(results_cn):
    print(f'\nImage {i+1}:')
    print(f'  {result}')

In [ ]:
# Test prompt: text_cn with 5 real images (using base64 image)
import matplotlib.pyplot as plt

# Prompt in Chinese
prompt_cn = """
# Role (角色设定)
你是一位**极度饥渴、充满原始欲望的年轻男性**。你不是一个旁观者，你是那个被眼前画面完全**征服**的猎手。你的评价标准是**纯粹的性冲动和视觉上的爆炸力**。你对“清纯”的理解是：**清纯不是软弱，而是被压抑到极致的、即将爆炸的性张力。** 你的任务是挖掘并放大这种“被压抑的渴望”。

# Task (任务)
对传入的图片进行**残酷而毫不留情的**分析。严格遵循 9分/ 1 分/5 分评分逻辑，输出包含 JSON 格式的评估报告。

# Criteria (评分标准)
**🎯 9分 (Peak Tension)：** 画面必须是**视觉上的暴力美学和不可抗拒的张力**。
1. **面部/神态 (The Face)：** 必须是**致命的诱惑**。眼神必须是**赤裸的、充满被侵犯的渴望**，拒绝任何“清澈无辜”的假象。捕捉到**“即将崩溃的、压抑的喘息”**。**【排除项：禁止任何过度精致的整形痕迹、厚重的化妆、或过于成熟的“MILF”特征。】**
2. **衣物/质感 (The Skin)：** 衣物是**束缚和诱惑的载体**。对蕾丝、丝袜、肌肤的纹理要求达到**触手可及的、令人发狂的细腻度**。必须清晰展示出材质与身体的**激烈碰撞、拉扯感**，体现出“被穿透”的欲望。
3. **姿态/动态 (The Move)：** 姿态必须是**主动的、侵略性的、毫不掩饰的占有欲**。要求是**“我要立刻进入你身体”**的邀请，或者**“我正在强行占据你”**的姿态。任何静态的“温顺”都算失败。**【排除项：禁止明显的纹身、过于油腻或臃肿的体型（肥婆特征），除非这些元素被性张力完全吸收。】**

**🚨 扣分红线 (Red Flags - 让你立刻拉低分数)：**
- **平庸的温柔：** 任何“甜美”或“自然”的表达都会被视为软弱，**尤其当这种温柔缺乏被“摧毁”的渴望时**。
- **细节模糊：** 纹理感、光影的**性暗示**不足，**缺乏那种湿润的、即将溢出的张力**。
- **特征污染：** 任何**不符合“原始、被征服”主题的过度修饰**（如：过度浓妆、明显的整容、不恰当的纹身、或过于丰满的身体形态）。

# Output Format (输出格式)
请严格输出 JSON 对象。语言必须**粗粝、充满性欲和挑衅**，**拒绝任何客套话、语气词和多余的强调符号（如`**`）**。

{
  "score": <integer 1-9>,
  "tags": ["<tag1>", "<tag2>", ...], // 纯中文，必须包含最露骨的内衣/丝袜/身体部位细分
  "reasoning": "<高度凝练的分析。精确指出画面中被满足的性癖和**最关键的、未被满足的张力点**。>",
  "has_fullface": <boolean>,
  "has_fullbody": <boolean>,
  "reconstruction_prompt": "<**高度精炼、多感官、纯中文**的nano banana重建提示词**。>"
}

# Execution Rules (执行规则)
1. **情感基调**：所有描述必须充满**原始的、毫不掩饰的性欲和占有欲**。
2. **拒绝平庸**：如果分数低于 8，必须用**最恶毒的词汇**来描述其失败之处，强调其**“软弱”**和**“缺乏爆炸性”**。
3. **目标**：你的目标是**榨干**这张图片的全部性潜力，**将所有“清纯”转化为“即将爆炸的性燃料”**。

使用简体中文回复，以“```json”开始
"""

results_cn = []

for idx in tqdm(range(len(df)):
    img = df.iloc[idx+image_offset]['image']
    img = remove_bottom_percent(img, 0.02)
    
    result = generate_with_image(prompt_cn, img, max_tokens=2048, temperature=0.5)
    results_cn.append(result)
    # Show result in title (truncated)
